# DATA 622: Homework 3
**Author:** Brett Allen (ballen3@umbc.edu)

**Date:** 02/21/2026

## Setup

In [1]:
!python -m pip install -q requests beautifulsoup4 nltk sentence-transformers scikit-learn

### Imports

In [2]:
import requests
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/home/ballen/anaconda3/envs/data-science/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-02-21 12:35:22.299914: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-21 12:35:22.311106: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771695322.324653   26243 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E000

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

### Configurations

In [3]:
# Use a custom User-Agent header to mimic a real browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

## 1. **Read the File**

Read the content of https://www.washingtonpost.com/world/2025/06/13/air-india-plane-crash-survivor-vishwash-kumar-ramesh/ into a Python variable. Load the first 700 characters.

In [4]:
url = "https://www.washingtonpost.com/world/2025/06/13/air-india-plane-crash-survivor-vishwash-kumar-ramesh/"

# timeout=(10, 30) means:
# - 10 seconds to establish a connection to the server
# - 30 seconds to wait for the server to send a response after the connection is established
response = requests.get(url, headers=headers, timeout=(10, 30), allow_redirects=True)
response.status_code

ReadTimeout: HTTPSConnectionPool(host='www.washingtonpost.com', port=443): Read timed out. (read timeout=30)

In [5]:
# Use an alternate link to the same article to work around the read timeout issue from the washingpost article
url = "https://www.bbc.com/news/articles/cp85zvne1m3o"
response = requests.get(url, headers=headers, timeout=(10, 30), allow_redirects=True)
response.status_code

200

In [6]:
# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Extract all paragraph tags
paragraphs = soup.find_all("p")

In [7]:
print(f"Number of paragraph tags: {len(paragraphs)}")

Number of paragraph tags: 48


In [8]:
# Inspect first 5 paragraph tags + content
paragraphs[:5]

[<p class="sc-9a00e533-0 eZyhnA">The sole survivor of the Air India plane crash, which killed 241 people on board, has said he feels like the "luckiest man" alive, but is also suffering physically and mentally.</p>,
 <p class="sc-9a00e533-0 eZyhnA">Viswashkumar Ramesh walked away from the wreckage of the London-bound flight in Ahmedabad in extraordinary scenes that amazed the world.</p>,
 <p class="sc-9a00e533-0 eZyhnA">He said it was a "miracle" he escaped but told how he has lost everything, as his younger brother Ajay was a few seats away on the flight and died in the crash in June.</p>,
 <p class="sc-9a00e533-0 eZyhnA">Since returning to his home in Leicester, Mr Ramesh has struggled with post-traumatic stress disorder (PTSD), his advisers said, and has been unable to speak to his wife and four-year-old son.</p>,
 <p class="sc-9a00e533-0 eZyhnA">Flames engulfed the Boeing 787 flight when it went down shortly after take-off in western India.</p>]

In [9]:
# Test getting text from the first paragraph tag
paragraphs[0].get_text(strip=True)

'The sole survivor of the Air India plane crash, which killed 241 people on board, has said he feels like the "luckiest man" alive, but is also suffering physically and mentally.'

In [10]:
# Join all paragraphs into a single text to represent the article
article_text = " ".join(p.get_text(strip=True) for p in paragraphs)

In [11]:
# Print the first 700 characters of the article text
print(article_text[:700])

The sole survivor of the Air India plane crash, which killed 241 people on board, has said he feels like the "luckiest man" alive, but is also suffering physically and mentally. Viswashkumar Ramesh walked away from the wreckage of the London-bound flight in Ahmedabad in extraordinary scenes that amazed the world. He said it was a "miracle" he escaped but told how he has lost everything, as his younger brother Ajay was a few seats away on the flight and died in the crash in June. Since returning to his home in Leicester, Mr Ramesh has struggled with post-traumatic stress disorder (PTSD), his advisers said, and has been unable to speak to his wife and four-year-old son. Flames engulfed the Boe


## 2. **Split the Text into Sentences**

Split the text into sentences using NLTK’s sentence tokenizer.

In [12]:
sents = sent_tokenize(article_text)
print(f"Number of sentences: {len(sents)}")

Number of sentences: 59


In [13]:
print(f"First 5 sentences:\n{sents[:5]}")

First 5 sentences:
['The sole survivor of the Air India plane crash, which killed 241 people on board, has said he feels like the "luckiest man" alive, but is also suffering physically and mentally.', 'Viswashkumar Ramesh walked away from the wreckage of the London-bound flight in Ahmedabad in extraordinary scenes that amazed the world.', 'He said it was a "miracle" he escaped but told how he has lost everything, as his younger brother Ajay was a few seats away on the flight and died in the crash in June.', 'Since returning to his home in Leicester, Mr Ramesh has struggled with post-traumatic stress disorder (PTSD), his advisers said, and has been unable to speak to his wife and four-year-old son.', 'Flames engulfed the Boeing 787 flight when it went down shortly after take-off in western India.']


## 3. **Load a Pre-trained Embedding Model**

Load a pre-trained sentence embedding model (such as all-MiniLM-L6-v2 from the sentence-transformers library or anyone of your choice).

Secondly, use TF/IDF to vectorize the first ten sentences.

In [14]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [15]:
# Use TF-IDF to vectorize the first 10 sentences
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sents[:10])

In [16]:
# Inspect the TF-IDF matrix
tfidf_matrix.shape

(10, 126)

## 4. **Embed Each Sentence**

Generate an embedding for each sentence.

Print the shape of the embedding for the first sentence.

In [17]:
sent_embeddings = embedding_model.encode(sents)

# Print the shape of the embedding for the first sentence
print(f"Shape of the embedding for the first sentence: {sent_embeddings[0].shape}")

Shape of the embedding for the first sentence: (384,)


In [18]:
# Shape of all sentences
sent_embeddings.shape

(59, 384)

In [19]:
# Inspect the first sentence embedding vector
sent_embeddings[0]

array([ 7.36028105e-02, -3.51374112e-02, -2.79700235e-02,  4.56262864e-02,
       -2.63813380e-02, -3.60681042e-02,  1.42283455e-01,  1.14764608e-02,
       -4.32141684e-02, -2.34003067e-02, -2.66981386e-02,  6.32638251e-03,
        1.09116081e-02,  3.01317386e-02, -3.69995385e-02, -9.50978976e-03,
        3.92625295e-03, -3.76653746e-02, -4.08886150e-02,  1.26212537e-01,
       -9.15686861e-02,  7.12781772e-02, -5.31179607e-02, -3.37470770e-02,
       -2.39398964e-02, -4.30694818e-02,  1.57120135e-02,  2.93858517e-02,
        5.17069362e-02, -3.30999419e-02,  6.40671700e-02, -4.00070772e-02,
       -6.71858862e-02,  1.23032858e-03,  4.08573709e-02, -3.60423140e-03,
       -8.56603310e-02, -4.22094576e-02,  4.05523274e-03, -8.18869695e-02,
       -3.73998936e-03, -5.59302270e-02,  1.22343889e-02,  6.49745837e-02,
       -3.77540514e-02, -6.94044381e-02, -9.63070691e-02, -7.69429058e-02,
        9.59102660e-02, -4.02240790e-02, -1.00580238e-01,  1.55912479e-02,
        8.75263289e-02, -

## 5. **Compute Similarity Between Sentences**

Compute the cosine similarity between the embeddings of the first and second sentences.

Print the similarity score.

In [20]:
help(cosine_similarity)

Help on function cosine_similarity in module sklearn.metrics.pairwise:

cosine_similarity(X, Y=None, dense_output=True)
    Compute cosine similarity between samples in X and Y.

    Cosine similarity, or the cosine kernel, computes similarity as the
    normalized dot product of X and Y:

    .. code-block:: text

        K(X, Y) = <X, Y> / (||X||*||Y||)

    On L2-normalized data, this function is equivalent to linear_kernel.

    Read more in the :ref:`User Guide <cosine_similarity>`.

    Parameters
    ----------
    X : {array-like, sparse matrix} of shape (n_samples_X, n_features)
        Input data.

    Y : {array-like, sparse matrix} of shape (n_samples_Y, n_features),             default=None
        Input data. If ``None``, the output will be the pairwise
        similarities between all samples in ``X``.

    dense_output : bool, default=True
        Whether to return dense output even when the input is sparse. If
        ``False``, the output is sparse if both input array

In [21]:
# Obtain the first and second sentence embeddings
sent_1_embedding = sent_embeddings[0]
sent_2_embedding = sent_embeddings[1]

In [22]:
sent_1_embedding.shape, sent_2_embedding.shape

((384,), (384,))

In [23]:
# Scikit-learn's cosine_similarity function expects 2D arrays, so we need to reshape the embeddings from (384,) to (1, 384)
# reshape(1, -1) means reshape to have 1 row and as many columns as needed (in this case, 384)
sent_1_embedding = sent_1_embedding.reshape(1, -1)
sent_2_embedding = sent_2_embedding.reshape(1, -1)

sent_1_embedding.shape, sent_2_embedding.shape

((1, 384), (1, 384))

In [24]:
# Now we can compute the cosine similarity between the two sentence embeddings
similarity = cosine_similarity(sent_1_embedding, sent_2_embedding)

In [25]:
similarity

array([[0.19589348]], dtype=float32)

In [26]:
print(f"Similarity between the first and second sentences: {similarity[0][0]:.4f}")

Similarity between the first and second sentences: 0.1959


In [27]:
# Take a look at the first and second sentences again to see if the similarity score makes sense
print("="*60)
print(f"Sentence 1: {sents[0]}")
print("="*60)
print(f"Sentence 2: {sents[1]}")
print("="*60)

Sentence 1: The sole survivor of the Air India plane crash, which killed 241 people on board, has said he feels like the "luckiest man" alive, but is also suffering physically and mentally.
Sentence 2: Viswashkumar Ramesh walked away from the wreckage of the London-bound flight in Ahmedabad in extraordinary scenes that amazed the world.
